In [0]:

BASE_DIR = '/Volumes/sales/landing/volumes/OrderPayments/'
CHECKPOINT_DIR = '/Volumes/sales/landing/volumes/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/OrderPayments'
CATALOG = 'SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'ORDERPAYMENTS'
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location:{BASE_DIR}')

In [0]:
orderpayments_schema = """order_id STRING, 
                 payment_sequential INT, 
                 payment_type STRING, 
                 payment_installments INT, 
                 payment_value DECIMAL(10,2)
                 """

In [0]:
orderpayments_df = (spark.read.format('csv').option('header','true').schema(orderpayments_schema).load(f'{BASE_DIR}')
           )
display(orderpayments_df.limit(3))
orderpayments_df.printSchema()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

orderpayments_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(orderpayments_schema)
                 .load(f'{BASE_DIR}')
        )
orderitem_trans_df = orderpayments_df.select(
    "order_id", 
    "payment_sequential", 
    "payment_type", 
    "payment_installments", 
    "payment_value",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

In [0]:
qry = (
    orderitem_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()